# 植物疾病识别模型：推理 Notebook

**作者**: Zihan

### 步骤 1 - （一次性工具）创建类别映射文件

**说明**: 此单元格用于将训练阶段生成的类别映射字典保存为标准 JSON 文件。只需运行一次，生成的 `class_map.json` 文件可与模型文件一起部署。

**操作流程**:
1. 从 Colab 训练 Notebook 复制 `idx_to_label` 字典
2. 清理数据类型（NumPy → Python 原生类型）
3. 保存为 JSON 格式

In [ ]:
# ========== 步骤 1：创建类别映射文件（一次性工具）==========
# import json
# import numpy as np

# # 1. 将你从 Colab 复制的字典粘贴在这里
# # 我们需要 import numpy as np 才能让 Python 理解 np.int64() 是什么
# original_map = {0: np.int64(0), 1: np.int64(5), 2: np.int64(6), ...}

# # 2. 清理数据，将其转换为纯粹的 Python 类型
# #   - 将 key 转换为字符串 (JSON 标准)
# #   - 将 value 从 np.int64 转换为普通的 int
# cleaned_map = {str(k): int(v) for k, v in original_map.items()}

# # 3. 定义输出文件名
# output_filename = "class_map.json"

# # 4. 使用 json 库将清理后的字典保存为格式正确的文件
# with open(output_filename, 'w') as f:
#     json.dump(cleaned_map, f, indent=4)

# print(f"✅ 成功创建了格式完全正确的 '{output_filename}' 文件！")

### 步骤 2 - 环境配置与路径设置

**目的**: 导入必要的库，配置模型路径、映射文件路径和计算设备。

**关键特性**:
- 使用 `NOTEBOOK_DIR` 动态获取脚本所在目录，确保路径稳健
- 所有配置集中在 `CFG` 类中，便于统一管理
- 自动检测 GPU/CPU 设备

In [ ]:
# ========== 步骤 2：环境配置与路径设置 ==========
import torch                                        # PyTorch 框架
import torch.nn.functional as F                     # 函数式接口（Softmax 等）
from torchvision import transforms                  # 图像变换
from PIL import Image                               # 图像处理
from pathlib import Path                            # 路径处理
import json, timm, warnings, os                     # JSON、模型库、警告、系统

warnings.filterwarnings('ignore')                   # 忽略不影响结果的警告

# ========== 配置信息 ==========
# --- 动态获取当前 Notebook 或脚本所在的目录 ---
# 确保总能从文件所在位置开始寻找，无论当前工作目录是什么
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path().resolve()


class CFG:
    """存放所有配置信息，方便统一修改"""
    
    # --- 路径配置（基于 NOTEBOOK_DIR，非常稳健）---
    MODEL_PATH     = NOTEBOOK_DIR / "model.pth"           # 训练好的模型权重文件
    CLASS_MAP_PATH = NOTEBOOK_DIR / "class_map.json"      # 索引→ID 的映射文件
    
    # --- 模型配置（必须与训练时完全一致）---
    MODEL_NAME     = 'swin_base_patch4_window7_224.ms_in1k'  # 模型名称
    IMAGE_SIZE     = 224                                     # 图像尺寸
    
    # --- 推理设备 ---
    DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 计算设备

# --- 打印最终绝对路径以供检查 ---
print(f"脚本/Notebook 所在目录：{NOTEBOOK_DIR}")
print(f"模型文件预期路径：{CFG.MODEL_PATH}")
print(f"映射文件预期路径：{CFG.CLASS_MAP_PATH}")

c:\Users\zyyin1\anaconda3\envs\python_for_ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


脚本/Notebook所在目录: E:\05_YZH_DS\02_Monash_DS\2025_S2_FIT5120_Industry_Experience_Studio_Project\06_main_project\03_github_submission\03_github_submission\2025-08-SDG13-Plant-X-Website\03_machine_learning_models\02_plant_disease_recognition
模型文件预期路径: E:\05_YZH_DS\02_Monash_DS\2025_S2_FIT5120_Industry_Experience_Studio_Project\06_main_project\03_github_submission\03_github_submission\2025-08-SDG13-Plant-X-Website\03_machine_learning_models\02_plant_disease_recognition\model.pth
映射文件预期路径: E:\05_YZH_DS\02_Monash_DS\2025_S2_FIT5120_Industry_Experience_Studio_Project\06_main_project\03_github_submission\03_github_submission\2025-08-SDG13-Plant-X-Website\03_machine_learning_models\02_plant_disease_recognition\class_map.json


### 步骤 3 - 加载类别映射字典

**目的**: 从 JSON 文件加载"翻译词典"（模型索引 → 真实疾病 ID）。

**说明**: 
- JSON 加载的 key 默认为字符串，需转换为整数以匹配 PyTorch 输出
- 包含错误处理，文件不存在或格式错误时给出明确提示

In [ ]:
# ========== 步骤 3：加载类别映射字典 ==========
def load_class_map(json_path):
    """
    从 JSON 文件加载 索引→ID 的映射字典
    
    参数:
        json_path: JSON 文件路径
    
    返回:
        映射字典 {int: int}，加载失败时返回 None
    """
    try:
        with open(json_path, 'r') as f:
            idx_to_label = json.load(f)
            # JSON 加载的 key 默认是字符串，需转为整数以匹配 PyTorch 输出
            idx_to_label = {int(k): v for k, v in idx_to_label.items()}
        print("✅ 翻译词典加载成功。")
        return idx_to_label
    except FileNotFoundError:
        print(f"🛑 错误：映射文件未找到于 '{json_path}'")
        return None
    except json.JSONDecodeError:
        print(f"🛑 错误：'{json_path}' 不是一个有效的 JSON 文件。")
        return None

### 步骤 4 - 加载训练好的模型

**目的**: 加载 Swin Transformer 模型架构并载入训练好的权重。

**关键点**:
- `pretrained=False`：不使用 ImageNet 预训练权重，加载本地权重
- `map_location=device`：确保跨设备兼容性（GPU/CPU 训练均可加载）
- `model.eval()`：切换到评估模式（禁用 Dropout 等训练专用层）

In [ ]:
# ========== 步骤 4：加载训练好的模型 ==========
def load_model(model_name, num_classes, model_path, device):
    """
    加载模型架构并载入训练好的权重
    
    参数:
        model_name: 模型名称（timm 格式）
        num_classes: 输出类别数
        model_path: 模型权重文件路径
        device: 计算设备
    
    返回:
        加载好的模型，加载失败时返回 None
    """
    try:
        # pretrained=False，因为我们要加载自己的本地权重
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
        # map_location=device 确保无论模型在 GPU 还是 CPU 上训练，都能正确加载
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)                                        # 移动到设备
        model.eval()                                            # 切换到评估模式
        print("✅ 模型加载成功。")
        return model
    except FileNotFoundError:
        print(f"🛑 错误：模型文件未找到于 '{model_path}'")
        return None
    except Exception as e:
        print(f"🛑 加载模型时发生未知错误：{e}")
        return None

In [ ]:
# ========== 定义图像预处理流程 ==========
# 推理时使用的图像转换，必须与训练时的验证/测试集转换完全一致
inference_transforms = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),                    # 调整尺寸
    transforms.ToTensor(),                                                  # 转换为 Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],                        # 归一化（ImageNet 均值）
                         std=[0.229, 0.224, 0.225])                         # 归一化（ImageNet 标准差）
])

### 步骤 5 - 定义图像预处理管道

**目的**: 定义推理时的图像转换流程。

**重要**: 必须与训练时的验证/测试集转换完全一致，包括：
- 图像尺寸调整（224×224）
- 转换为 Tensor
- 归一化（ImageNet 均值和标准差）

In [ ]:
# ========== 步骤 5：核心预测函数 ==========
def predict_top3(model, image_path, transforms, idx_to_label_map, device):
    """
    对单张图片进行 Top-3 预测，只输出 ID 和概率
    
    参数:
        model: 加载好的模型
        image_path: 输入图像路径
        transforms: 图像预处理管道
        idx_to_label_map: 索引→ID 的映射字典
        device: 计算设备
    
    返回:
        Top-3 预测结果列表 [{"predicted_id": int, "probability": str}, ...]
        出错时返回错误信息字符串
    """
    try:
        # 加载并预处理图片
        image = Image.open(image_path).convert("RGB")
        # .unsqueeze(0) 增加 batch 维度，模型需要 [B, C, H, W] 形状的输入
        image_tensor = transforms(image).unsqueeze(0).to(device)
    except FileNotFoundError:
        return f"🛑 错误：图片文件未找到于 '{image_path}'"
    except Exception as e:
        return f"🛑 处理图片时发生错误：{e}"

    # 关闭梯度计算，加速推理并节省显存
    with torch.no_grad():
        logits = model(image_tensor)                                    # 前向传播

    # 应用 Softmax 函数将 logits 转换为概率
    probabilities = F.softmax(logits, dim=1)
    
    # 获取概率最高的 3 个值和它们对应的索引
    top3_probs, top3_indices = torch.topk(probabilities, 3, dim=1)

    # 将 Tensor 从 GPU 移动到 CPU，并转换为 NumPy 数组
    top3_probs   = top3_probs.squeeze().cpu().numpy()
    top3_indices = top3_indices.squeeze().cpu().numpy()
    
    results = []
    for i in range(3):
        class_idx   = top3_indices[i]
        class_id    = idx_to_label_map[class_idx]                       # 索引→ID 翻译
        prob        = top3_probs[i]
        
        results.append({
            "predicted_id": int(class_id),                              # 确保 ID 是整数
            "probability": f"{prob:.2%}"                                # 格式化为百分比
        })
        
    return results

### 步骤 6 - 执行推理并显示结果

**目的**: 对单张测试图像进行 Top-3 预测，输出疾病 ID 和概率。

**输出格式**:  
`病害 ID: 数字 | 概率：xx.xx%`  

**使用说明**:   
修改 `TEST_IMAGE_PATH` 为本地测试图片路径后运行。

In [ ]:
# ========== 步骤 6：主执行入口 ==========
if __name__ == '__main__':
    print("--- 开始植物病害识别推理 ---")
    print(f"使用设备：{CFG.DEVICE}")
    
    # 步骤 1: 加载翻译词典
    print("\n[1/3] 正在加载标签映射表...")
    idx_to_label = load_class_map(CFG.CLASS_MAP_PATH)
    
    # 步骤 2: 加载模型
    if idx_to_label is not None:
        print("\n[2/3] 正在加载模型...")
        num_classes = len(idx_to_label)                                 # 类别数必须与词典大小一致
        model = load_model(CFG.MODEL_NAME, num_classes, CFG.MODEL_PATH, CFG.DEVICE)
    else:
        model = None

    # 步骤 3: 执行预测
    if model is not None:
        print("\n[3/3] 正在执行预测...")
        # !! 需要你指定一张本地图片的路径 !!
        TEST_IMAGE_PATH = "test_images/下载.jfif"                       # <--- 修改为本地测试图片路径

        if Path(TEST_IMAGE_PATH).exists():
            top3_predictions = predict_top3(model, TEST_IMAGE_PATH, inference_transforms,
                                            idx_to_label, CFG.DEVICE)
            
            print("\n" + "="*30)
            print("--- 预测结果 (Top 3) ---")
            print(f"图片：{TEST_IMAGE_PATH}")
            print("="*30)
            for pred in top3_predictions:
                print(f"病害 ID: {pred['predicted_id']:<5} | 概率：{pred['probability']}")
        else:
            print(f"🛑 错误：测试图片 '{TEST_IMAGE_PATH}' 不存在。请修改路径后重试。")

--- 开始植物病害识别推理 ---
使用设备: cuda

[1/3] 正在加载标签映射表...
✅ 翻译词典加载成功。

[2/3] 正在加载模型...
✅ 模型加载成功。

[3/3] 正在执行预测...

--- 预测结果 (Top 3) ---
图片: test_images/下载.jfif
病害ID: 6     | 概率: 85.83%
病害ID: 22    | 概率: 5.06%
病害ID: 61    | 概率: 3.54%
